<a href="https://colab.research.google.com/github/pratik111111/Medical_Model_Q-A/blob/main/M_Q%26A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import json
import torch
import numpy as np
import re
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score
from collections import Counter
from dataclasses import dataclass
from transformers.tokenization_utils_base import PreTrainedTokenizerBase
from typing import Optional, Union
import torch

# ==============================
# 3. LOAD DATA
# ==============================
with open("medical_meadow_medqa.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print("Total raw samples:", len(raw_data))

Total raw samples: 10178


In [17]:
# ==============================
# 4. PARSE OPTIONS FROM TEXT
# ==============================
label_map = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
processed_data = []

for item in raw_data:
    try:
        full_input = item["input"]
        answer = item["output"]
        label_letter = answer.split(":")[0].strip()

        if label_letter not in label_map:
            continue

        # Extract question (before the options dict)
        question_match = re.split(r"\n\s*\{", full_input)
        question = question_match[0].strip()

        # Extract options dict from input
        options_match = re.search(r"\{([^}]+)\}", full_input)
        if not options_match:
            continue

        options_str = "{" + options_match.group(1) + "}"
        # safely parse: 'A': 'text', ...
        options = {}
        for match in re.finditer(r"'([ABCDE])':\s*'([^']+)'", options_str):
            options[match.group(1)] = match.group(2)

        if len(options) < 2:
            continue

        processed_data.append({
            "question": question,
            "option_a": options.get("A", ""),
            "option_b": options.get("B", ""),
            "option_c": options.get("C", ""),
            "option_d": options.get("D", ""),
            "option_e": options.get("E", ""),
            "label": int(label_map[label_letter])
        })

    except Exception as e:
        continue

print("Processed samples:", len(processed_data))
print("Label distribution:", Counter(d["label"] for d in processed_data))
print("\nSample item:")
print("Q:", processed_data[0]["question"][:100])
print("A:", processed_data[0]["option_a"])
print("Label:", processed_data[0]["label"])

Processed samples: 10171
Label distribution: Counter({1: 2183, 2: 2084, 3: 2035, 0: 2030, 4: 1839})

Sample item:
Q: Q:A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She state
A: Ampicillin
Label: 4


In [18]:
# ==============================
# 5. CREATE DATASET
# ==============================
dataset = Dataset.from_list(processed_data)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

In [19]:
# ==============================
# 6. LOAD TOKENIZER & MODEL
# ==============================
model_name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# ==============================
# 7. TOKENIZE FOR MULTIPLE CHOICE
# ==============================
option_keys = ["option_a", "option_b", "option_c", "option_d", "option_e"]

def tokenize_mc(batch):
    questions = batch["question"]

    # For each of 5 options, tokenize [question, option] pair
    all_encodings = [
        tokenizer(
            questions,
            batch[opt],
            padding="max_length",
            truncation=True,
            max_length=256
        )
        for opt in option_keys
    ]

    # Stack: shape = [batch, 5, seq_len]
    return {
        "input_ids": [
            [all_encodings[i]["input_ids"][j] for i in range(5)]
            for j in range(len(questions))
        ],
        "attention_mask": [
            [all_encodings[i]["attention_mask"][j] for i in range(5)]
            for j in range(len(questions))
        ],
        "label": batch["label"]
    }

tokenized = dataset.map(tokenize_mc, batched=True,
                        remove_columns=dataset["train"].column_names)
tokenized.set_format(type="torch")

Map:   0%|          | 0/9153 [00:00<?, ? examples/s]

Map:   0%|          | 0/1018 [00:00<?, ? examples/s]

In [20]:
# ==============================
# 8. LOAD MODEL
# ==============================
model = AutoModelForMultipleChoice.from_pretrained(
    model_name,
    num_labels=5
)

# ==============================
# 9. TRAINING CONFIG
# ==============================
training_args = TrainingArguments(
    output_dir="./results_mc",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,   # MC uses 5x memory
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=torch.cuda.is_available()
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	

In [21]:
# ==============================
# 10. METRICS
# ==============================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

# ==============================
# 11. TRAINER
# ==============================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [22]:
# ==============================
# 12. TRAIN
# ==============================
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.609032,1.597407,0.252456
2,1.555935,1.555272,0.296660
3,1.389894,1.658594,0.311395
4,1.071112,2.038870,0.308448
5,0.854566,2.457324,0.319253


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=11445, training_loss=1.3030122356156482, metrics={'train_runtime': 3310.737, 'train_samples_per_second': 13.823, 'train_steps_per_second': 3.457, 'total_flos': 3.01029233369472e+16, 'train_loss': 1.3030122356156482, 'epoch': 5.0})

In [23]:
# ==============================
# 13. EVALUATE
# ==============================
results = trainer.evaluate()
print("Evaluation Results:", results)

Evaluation Results: {'eval_loss': 2.4573237895965576, 'eval_accuracy': 0.3192534381139489, 'eval_runtime': 19.5876, 'eval_samples_per_second': 51.972, 'eval_steps_per_second': 13.018, 'epoch': 5.0}


In [24]:
# ==============================
# 14. SAVE
# ==============================
model.save_pretrained("./medical_model_mc")
tokenizer.save_pretrained("./medical_model_mc")
print("✅ Model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved!


In [27]:
# ==============================
# 15. PREDICT
# ==============================
reverse_map = {v: k for k, v in label_map.items()}

def predict(item):
    device = next(model.parameters()).device
    question = item["question"]
    options = [item[k] for k in option_keys]

    encodings = tokenizer(
        [question] * 5,
        options,
        padding="max_length",
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    input_ids = encodings["input_ids"].unsqueeze(0).to(device)
    attention_mask = encodings["attention_mask"].unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    pred = torch.argmax(outputs.logits, dim=1).item()
    return reverse_map[pred]
    # Test on first sample
sample = processed_data[5]
print("Prediction:", predict(sample))
print("Correct:", raw_data[5]["output"])


Prediction: C
Correct: C: Scorpion sting
